# Project V — Computational Discovery
## Milestone 1: Feature-Space Design

This notebook starts Project V of the Gaia–LAMOST Galactic Archaeology research sequence.

Project V focuses on machine-learning assisted stellar population clustering using the larger Gaia–LAMOST velocity-feature table produced in Project I.

The goal of this milestone is not to claim any astrophysical discovery. Instead, it defines the input dataset, inspects available features, and prepares several feature-space groups for later clustering experiments.

## Research Question

Can unsupervised machine learning recover interpretable stellar population structure from Gaia–LAMOST chemo-kinematic features?

## Primary Input Dataset

The primary Project V input table is:

`../data/processed/gaia_lamost_larger_velocity_features.csv`

This table contains Gaia astrometry and photometry, LAMOST spectroscopic parameters, derived kinematic quantities, Galactic coordinates, and Galactocentric velocity features.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

In [ ]:
DATA_PATH = Path('../data/processed/gaia_lamost_larger_velocity_features.csv')

df = pd.read_csv(DATA_PATH)
df.shape

In [ ]:
df.head()

## Available Columns

In [ ]:
for col in df.columns:
    print(col)

## Candidate Interpretation Columns

The following Project I columns are useful for interpretation, but should not be used as input labels for unsupervised clustering.

In [ ]:
interpretation_cols = [
    'metallicity_group',
    'high_vtan_candidate',
    'metal_poor_candidate',
    'chemo_kinematic_candidate',
]

for col in interpretation_cols:
    if col in df.columns:
        print(f'\
{col}')
        print(df[col].value_counts(dropna=False))

## Feature-Space Definitions

Several feature spaces are defined for later clustering experiments. These groups separate photometric, chemical, kinematic, and combined chemo-kinematic information.

In [ ]:
feature_spaces = {
    'photometric_astrometric': [
        'bp_rp',
        'absolute_g_mag',
        'distance_pc',
        'pm_total',
        'reduced_pm_g',
    ],
    'chemical_stellar': [
        'feh',
        'teff',
        'logg',
    ],
    'local_kinematic': [
        'tangential_velocity_kms',
        'rv',
    ],
    'galactocentric_velocity': [
        'galcen_vx_kms',
        'galcen_vy_kms',
        'galcen_vz_kms',
        'galcen_vtot_kms',
    ],
    'combined_chemo_kinematic': [
        'feh',
        'bp_rp',
        'absolute_g_mag',
        'tangential_velocity_kms',
        'rv',
        'galcen_vx_kms',
        'galcen_vy_kms',
        'galcen_vz_kms',
        'galcen_vtot_kms',
    ],
}

feature_spaces

## Feature Availability and Missingness

Before clustering, each feature space should be checked for missing values and valid sample size.

In [ ]:
summary_rows = []

for name, cols in feature_spaces.items():
    missing_cols = [col for col in cols if col not in df.columns]
    available_cols = [col for col in cols if col in df.columns]
    complete_rows = df[available_cols].dropna().shape[0] if available_cols else 0
    summary_rows.append({
        'feature_space': name,
        'n_features_defined': len(cols),
        'n_features_available': len(available_cols),
        'missing_columns': ', '.join(missing_cols),
        'complete_rows': complete_rows,
    })

feature_space_summary = pd.DataFrame(summary_rows)
feature_space_summary

## Basic Numeric Summary

In [ ]:
numeric_cols = sorted(set(sum(feature_spaces.values(), [])))
numeric_cols = [col for col in numeric_cols if col in df.columns]

df[numeric_cols].describe().T

## Save Project V Feature-Space Summary

This table records which feature spaces are ready for later clustering experiments.

In [ ]:
OUTPUT_PATH = Path('../data/processed/project_v_feature_space_summary.csv')
feature_space_summary.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH

## Milestone 1 Notes

- Project V will use unsupervised methods only at this stage.
- Project I candidate flags will be used only for post-hoc interpretation.
- Clustering results should be interpreted as feature-space structures, not confirmed Galactic substructures.
- Later milestones will compare PCA, UMAP, DBSCAN/HDBSCAN, and Gaussian Mixture Models.